In [27]:
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding

import numpy as np

In [2]:
qd_client = QdrantClient("http://localhost:6333")

In [3]:
embedding_model_name = "jinaai/jina-embeddings-v2-small-en"

In [4]:
model = TextEmbedding(model_name=embedding_model_name)

## Q1. Embedding the query

In [5]:
query = "I just discovered the course. Can I join now?"

In [6]:
query_vector = model.embed(query)

In [7]:
query_vector = next(iter(query_vector))

In [8]:
query_vector.min()

np.float64(-0.11726373885183883)

## Q2. Cosine similarity with another vector

In [10]:
doc = "Can I still join the course after the start date?"

In [11]:
doc_vector = model.embed(doc)

In [12]:
doc_vector = next(iter(doc_vector))

In [13]:
query_vector.dot(doc_vector)

np.float64(0.9008528895674548)

## Q3. Ranking by cosine

In [ ]:
documents = [
    {
        "text": "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
        "section": "General course-related questions",
        "question": "Course - Can I still join the course after the start date?",
        "course": "data-engineering-zoomcamp",
    },
    {
        "text": "Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.",
        "section": "General course-related questions",
        "question": "Course - Can I follow the course after it finishes?",
        "course": "data-engineering-zoomcamp",
    },
    {
        "text": "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
        "section": "General course-related questions",
        "question": "Course - When will the course start?",
        "course": "data-engineering-zoomcamp",
    },
    {
        "text": "You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.",
        "section": "General course-related questions",
        "question": "Course - What can I do before the course starts?",
        "course": "data-engineering-zoomcamp",
    },
    {
        "text": "Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.",
        "section": "General course-related questions",
        "question": "How can we contribute to the course?",
        "course": "data-engineering-zoomcamp",
    },
]

In [ ]:
docs_embeddings = []

documents_qas = [document["text"] for document in documents]
for embedding in model.embed(documents=documents_qas):
    docs_embeddings.append(embedding.dot(query_vector))

docs_embeddings = np.array(docs_embeddings)

In [39]:
np.argsort(-docs_embeddings)

array([1, 2, 0, 4, 3])

## Q4. Ranking by cosine, version two

In [ ]:
docs_embeddings = []

documents_qas = [
    document["question"] + " " + document["text"] for document in documents
]
for embedding in model.embed(documents=documents_qas):
    docs_embeddings.append(embedding.dot(query_vector))

docs_embeddings = np.array(docs_embeddings)

In [42]:
docs_embeddings

array([0.85145432, 0.84365942, 0.8408287 , 0.7755158 , 0.80860078])

In [41]:
np.argsort(-docs_embeddings)

array([0, 1, 2, 4, 3])

## Q5. Selecting the embedding model

In [ ]:
minimum_dim = float("inf")

for model_info in TextEmbedding.list_supported_models():
    minimum_dim = min(minimum_dim, model_info["dim"])

In [62]:
minimum_dim

384

## Q6. Indexing with qdrant (2 points)

In [ ]:
import requests

docs_url = "https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json"
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()


documents = []

for course in documents_raw:
    course_name = course["course"]
    if course_name != "machine-learning-zoomcamp":
        continue

    for doc in course["documents"]:
        doc["course"] = course_name
        documents.append(doc)

In [67]:
collection_name = "qas_m2_hw"
embedding_mode_name = "BAAI/bge-small-en"
EMBEDDING_DIMENSIONALITY = 384

In [ ]:
qd_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY, distance=models.Distance.COSINE
    ),
)

True

In [69]:
qd_client.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=idx,
            vector=models.Document(
                text=document["question"] + " " + document["text"], 
        		model=embedding_mode_name
            ),
        )
        
		for idx, document in enumerate(documents)
    ],
)

Fetching 5 files: 100%|██████████| 5/5 [02:37<00:00, 31.47s/it]


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [70]:
result = qd_client.query_points(
  collection_name=collection_name,
  query=models.Document(
    text=query,
    model=embedding_mode_name
  )
)

In [77]:
result.points[0].score

0.8703172